In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, accuracy_score

## Data Preparation

In [2]:
df = pd.read_csv('data/titanic.csv')
print("Data loaded successfully.") 
display(df.head())

Data loaded successfully.


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1, inplace=True)

df.drop('Embarked', axis=1, inplace=True)

In [4]:
age_1 = df[df['Pclass'] == 1]['Age'].median()
age_2 = df[df['Pclass'] == 2]['Age'].median()
age_3 = df[df['Pclass'] == 3]['Age'].median()

def fill_age(row):
    if pd.isnull(row['Age']):
        if row['Pclass'] == 1:
            return age_1
        if row['Pclass'] == 2:
            return age_2
        return age_3
    return row['Age']

df['Age'] = df.apply(fill_age, axis=1)

In [5]:
# Convert 'Sex' from string to numeric values
def fill_sex(sex):
    if sex == 'male':
        return 1
    return 0 

df['Sex'] = df['Sex'].apply(fill_sex)

## Data Splitting

In [6]:
X = df.drop('Survived', axis=1)

y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

In [7]:
sc = StandardScaler()

X_train = sc.fit_transform(X_train)

X_test = sc.transform(X_test)

In [8]:
classifier = KNeighborsClassifier(n_neighbors=5)

classifier.fit(X_train, y_train)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.Doesn't affect :meth:`fit` method.",None


In [9]:
y_pred = classifier.predict(X_test)

print('Percentage of correctly predicted outcomes:', accuracy_score(y_test, y_pred) * 100)

print('\nConfusion matrix:')
print(confusion_matrix(y_test, y_pred))

Percentage of correctly predicted outcomes: 79.37219730941703

Confusion matrix:
[[114  20]
 [ 26  63]]


In [10]:
# Experiment 1: Varying K
print("--- RESULTS: EXPERIMENT 1 ---")

# List of k values to test
k_values = [3, 5, 7, 9]

for k in k_values:
    knn_exp1 = KNeighborsClassifier(n_neighbors=k)
    
    knn_exp1.fit(X_train, y_train)
    
    y_pred_exp1 = knn_exp1.predict(X_test)
    acc = accuracy_score(y_test, y_pred_exp1) * 100
    
    print(f"k = {k} -> Accuracy: {acc:.2f}%")

--- RESULTS: EXPERIMENT 1 ---
k = 3 -> Accuracy: 78.92%
k = 5 -> Accuracy: 79.37%
k = 7 -> Accuracy: 77.58%
k = 9 -> Accuracy: 78.92%


In [11]:
# Experiment 2: Without Scaling
print("\n--- RESULTS: EXPERIMENT 2 ---")

X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(X, y, test_size=0.25, random_state=42)

knn_raw = KNeighborsClassifier(n_neighbors=5)
knn_raw.fit(X_train_raw, y_train_raw)

y_pred_raw = knn_raw.predict(X_test_raw)
acc_raw = accuracy_score(y_test_raw, y_pred_raw) * 100

print(f"Accuracy WITH Scaling (k=5):    {accuracy_score(y_test, classifier.predict(X_test))*100:.2f}%")
print(f"Accuracy WITHOUT Scaling (k=5): {acc_raw:.2f}%")


--- RESULTS: EXPERIMENT 2 ---
Accuracy WITH Scaling (k=5):    79.37%
Accuracy WITHOUT Scaling (k=5): 70.40%


In [13]:
# Experiment 3: New Features
print("\n--- RESULTS: EXPERIMENT 3 ---")

# Reload Embarked column
df_new = pd.read_csv('data/titanic.csv')

df_new['Sex'] = df_new['Sex'].apply(lambda x: 1 if x == 'male' else 0)

age_1_new = df_new[df_new['Pclass'] == 1]['Age'].median()
age_2_new = df_new[df_new['Pclass'] == 2]['Age'].median()
age_3_new = df_new[df_new['Pclass'] == 3]['Age'].median()

def fill_age_new(row):
    if pd.isnull(row['Age']):
        if row['Pclass'] == 1:
            return age_1_new
        if row['Pclass'] == 2:
            return age_2_new
        return age_3_new
    return row['Age']

df_new['Age'] = df_new.apply(fill_age_new, axis=1)
df_new.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1, inplace=True)

df_new['FamilySize'] = df_new['SibSp'] + df_new['Parch'] + 1

df_new['Embarked'] = df_new['Embarked'].fillna('S') # Fill missing with 'S'
# One-Hot Encoding for Embarked
df_new = pd.get_dummies(df_new, columns=['Embarked'], drop_first=True)

X_new = df_new.drop('Survived', axis=1)
y_new = df_new['Survived']

X_train_new, X_test_new, y_train_new, y_test_new = train_test_split(X_new, y_new, test_size=0.25, random_state=42)

sc_new = StandardScaler()
X_train_new = sc_new.fit_transform(X_train_new)
X_test_new = sc_new.transform(X_test_new)

knn_new = KNeighborsClassifier(n_neighbors=5)
knn_new.fit(X_train_new, y_train_new)

acc_new = accuracy_score(y_test_new, knn_new.predict(X_test_new)) * 100
print(f"Accuracy with New Features (FamilySize, Embarked): {acc_new:.2f}%")


--- RESULTS: EXPERIMENT 3 ---
Accuracy with New Features (FamilySize, Embarked): 80.72%
